In [ ]:
import re
import nbformat as nbf

nb = nbf.v4.new_notebook()
cells = []

def md(text):
    cells.append(nbf.v4.new_markdown_cell(text))

def code(text):
    text = text.strip('\n')
    if text.strip():
        cells.append(nbf.v4.new_code_cell(text))

### ------------------ Portada ------------------
### Pipeline completo: Feature Engineering -> Entrenamiento -> Despliegue
#### TFM RMS — Hotel independiente en Baiona, Galicia

Este notebook encadena todo el flujo de un tirón:

1. **Carga** de las reservas (CSV en Drive, carpeta `BBDD_Modelo`)
2. **Feature engineering** (con los 3 fixes aplicados — ver notas en cada celda)
3. **Entrenamiento** del modelo XGBoost (con el fix del gráfico Predicción vs Real)
4. **Export** de `modelo_ocupacion.pkl` + `variables_modelo.pkl`
5. **Despliegue automático** a Cloud Functions vía `gcloud` (sin salir de Colab)
6. **Obtención de la URL** final de la function

**Fixes aplicados respecto a tus scripts originales:**
- ❌ `reservas_vigentes` referenciada antes de existir (NameError) → eliminado (código muerto, no se usaba después)
- ❌ Filtro Top-10 de `Canal`/`Tipo` usaba por error `top_paises` → corregido a `top_canal`/`top_tipo`
- ❌ Reservas *Canceladas* contaban como ocupación real (podía superar el 100%) → ahora se filtra a `Confirmada` + `No Show`
- ❌ `pred_test`/`y_test` no existían en el gráfico Predicción vs Real (NameError) → usa `pred_test_final`/`y_test_final`

#### 0. Configuración
Ajusta aquí las rutas y parámetros del proyecto GCP. Todo lo demás usa estas variables.

In [ ]:
# --- Rutas de Drive ---
CARPETA_RESERVAS = '/content/drive/MyDrive/Master/TFM/BBDD_Modelo/'        # CSVs de reservas (input)
CARPETA_MODELO   = '/content/drive/MyDrive/Master/TFM/Modelos TFM/'  # dataset_modelo.csv + .pkl (output)

# --- Parámetros de negocio ---
CAPACIDAD_HOTEL = 75
HORIZONTES = [7, 14, 30]

# --- Proyecto GCP / Cloud Function ---
GCP_PROJECT_ID   = 'long-classifier-498316-k1'
GCP_REGION       = 'europe-west1'
FUNCTION_NAME    = 'predecir-demanda'
FUNCTION_ENTRY   = 'predecir'          # debe coincidir con @functions_framework.http def predecir(...)
FUNCTION_RUNTIME = 'python311'
FUNCTION_MEMORY  = '512Mi'
FUNCTION_TIMEOUT = '60s'

# Carpeta local (dentro de la sesión de Colab) donde se prepara el paquete a desplegar
CARPETA_DEPLOY = '/content/deploy_predecir_demanda'

### 1. Montaje de Google Drive Con comprobación de seguridad

In [ ]:

import os

# Si /content/drive ya existe como carpeta local NO montada (residuo de una sesión anterior), la limpiamos
# antes de montar — pero solo si NO es ya un punto de montaje real, para no borrar nada de Drive por error.
if os.path.exists('/content/drive') and not os.path.ismount('/content/drive'):
    import shutil
    print("Aviso: /content/drive existe pero no es un montaje real de Drive. Se limpia antes de montar.")
    shutil.rmtree('/content/drive')

from google.colab import drive
drive.mount('/content/drive')

assert os.path.ismount('/content/drive'), "Drive NO está montado — repite drive.mount() y autoriza el acceso"
print("Drive montado correctamente ✅")

ValueError: mount failed

### 2. Instalación de librerías



In [ ]:
!pip -q install xgboost shap holidays

import warnings
warnings.filterwarnings("ignore")

import time
import random
import platform
from datetime import datetime, timedelta

import numpy as np
import pandas as pd
import joblib

import matplotlib.pyplot as plt
import matplotlib.ticker as mtick

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from xgboost import XGBRegressor

import shap
import holidays

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
random.seed(RANDOM_STATE)

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
plt.rcParams["figure.figsize"] = (10, 5)

print("Librerías cargadas correctamente.")

### ---------------------------- BLOQUE 1 — Carga del dataset de reservas ----------------------------

In [ ]:
import glob

ficheros = sorted(glob.glob(f'{CARPETA_RESERVAS}/*.csv'))
if not ficheros:
    raise FileNotFoundError(f'No se encontraron CSV en {CARPETA_RESERVAS}')

df = pd.concat([pd.read_csv(f) for f in ficheros], ignore_index=True)

print("Dataset cargado correctamente.")
print(f"Archivos encontrados: {len(ficheros)}")
for f in ficheros:
    print(f"  - {f.split('/')[-1]}")
print(f"Filas totales: {len(df)}")
""")
code("""
print(f"Filas..................... {df.shape[0]:,}")
print(f"Columnas.................. {df.shape[1]}")
memoria = df.memory_usage(deep=True).sum() / 1024**2
print(f"Memoria................... {memoria:.2f} MB")
display(df.head())
display(df.dtypes)

### ---------------------------- BLOQUE 2 — Auditoría y calidad del dato ----------------------------

In [ ]:
columnas_fecha = ["FechaReserva", "CheckIn", "CheckOut"]
for col in columnas_fecha:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], errors="coerce")
print("Conversión de fechas finalizada.")

print("="*70)
print("AUDITORÍA DEL DATASET")
print("="*70)
print(f"Número de registros      : {len(df):,}")
print(f"Número de columnas       : {df.shape[1]}")
print(f"Duplicados               : {df.duplicated().sum():,}")
if "CheckIn" in df.columns:
    print(f"Periodo: {df['CheckIn'].min().date()} -> {df['CheckIn'].max().date()}")

nulos = pd.DataFrame({"Nulos": df.isnull().sum(), "%": round(df.isnull().mean()*100, 2)})
nulos = nulos[nulos["Nulos"] > 0].sort_values("Nulos", ascending=False)
print("VALORES NULOS")
print("No existen valores nulos." if len(nulos) == 0 else "")
if len(nulos) > 0:
    display(nulos)

errores = {}
if {"CheckIn", "CheckOut"}.issubset(df.columns):
    errores["CheckOut < CheckIn"] = (df["CheckOut"] < df["CheckIn"]).sum()
if "Noches" in df.columns:
    errores["Noches <=0"] = (df["Noches"] <= 0).sum()
if "ADR" in df.columns:
    errores["ADR negativo"] = (df["ADR"] < 0).sum()
    errores["ADR >1000"] = (df["ADR"] > 1000).sum()
if "Adultos" in df.columns:
    errores["Adultos <=0"] = (df["Adultos"] <= 0).sum()
if "Ninos" in df.columns:
    errores["Niños negativos"] = (df["Ninos"] < 0).sum()

print("DETECCIÓN DE ERRORES")
display(pd.DataFrame.from_dict(errores, orient="index", columns=["Registros"]))



### ---------------------------- BLOQUE 3 — Feature engineering ----------------------------
Creación del dataset de modelado

In [ ]:
df_modelo = df.copy()
print(f"Registros originales : {len(df_modelo):,}")

antes = len(df_modelo)
df_modelo = df_modelo.drop_duplicates()
print(f"Duplicados eliminados : {antes - len(df_modelo)}")

antes = len(df_modelo)
df_modelo = df_modelo[df_modelo["CheckOut"] > df_modelo["CheckIn"]]
df_modelo = df_modelo[df_modelo["Noches"] > 0]
df_modelo = df_modelo[df_modelo["ADR"] >= 0]
df_modelo = df_modelo[df_modelo["Adultos"] > 0]
print(f"Registros eliminados por reglas de limpieza : {antes - len(df_modelo)}")

df_modelo["LeadTime"] = (df_modelo["CheckIn"] - df_modelo["FechaReserva"]).dt.days
df_modelo["LeadTime"] = df_modelo["LeadTime"].clip(lower=0)
display(df_modelo["LeadTime"].describe())

df_modelo["Año"] = df_modelo["CheckIn"].dt.year
df_modelo["Mes"] = df_modelo["CheckIn"].dt.month
df_modelo["Semana"] = df_modelo["CheckIn"].dt.isocalendar().week.astype(int)
df_modelo["DiaMes"] = df_modelo["CheckIn"].dt.day
df_modelo["DiaSemana"] = df_modelo["CheckIn"].dt.dayofweek
df_modelo["FinSemana"] = (df_modelo["DiaSemana"] >= 5).astype(int)
print("Variables temporales creadas.")

es = holidays.ES()
df_modelo["Festivo"] = df_modelo["CheckIn"].dt.date.isin(es).astype(int)
print(df_modelo["Festivo"].value_counts())

df_modelo["Mes_sin"] = np.sin(2*np.pi*df_modelo["Mes"]/12)
df_modelo["Mes_cos"] = np.cos(2*np.pi*df_modelo["Mes"]/12)
df_modelo["DiaSemana_sin"] = np.sin(2*np.pi*df_modelo["DiaSemana"]/7)
df_modelo["DiaSemana_cos"] = np.cos(2*np.pi*df_modelo["DiaSemana"]/7)
print("Variables cíclicas creadas.")

antes = len(df_modelo)
df_modelo = df_modelo[df_modelo["Estado"].isin(["Confirmada", "No Show"])].copy()
print(f"Registros excluidos por Estado (Cancelada): {antes - len(df_modelo)}")
print(f"Registros restantes: {len(df_modelo):,}")

CAPACIDAD_USADA_HOTEL = df_modelo["Habitacion"].nunique()
print(f"Capacidad estimada del hotel (habitaciones con reservas): {CAPACIDAD_USADA_HOTEL}")
print(f"Capacidad configurada (CAPACIDAD_HOTEL): {CAPACIDAD_HOTEL}")
print(f"Horizontes: {HORIZONTES}")

ocupacion_final = (df_modelo.groupby("CheckIn")
                    .agg(habitaciones_final=("Habitacion", "nunique"), adr_final=("ADR", "mean"))
                    .reset_index())
ocupacion_final["ocupacion_final"] = ocupacion_final["habitaciones_final"] / CAPACIDAD_HOTEL * 100
display(ocupacion_final.head())

top_paises = df_modelo["Pais"].value_counts().nlargest(10).index
df_modelo["Pais"] = df_modelo["Pais"].where(df_modelo["Pais"].isin(top_paises), "Otros")

top_canal = df_modelo["Canal"].value_counts().nlargest(10).index
df_modelo["Canal"] = df_modelo["Canal"].where(df_modelo["Canal"].isin(top_canal), "Otros")

top_tipo = df_modelo["Tipo"].value_counts().nlargest(10).index
df_modelo["Tipo"] = df_modelo["Tipo"].where(df_modelo["Tipo"].isin(top_tipo), "Otros")

print("Canal:", df_modelo["Canal"].unique())
print("Tipo:", df_modelo["Tipo"].unique())
print("Pais:", df_modelo["Pais"].unique())

dataset_modelo = []
fechas = sorted(df_modelo["CheckIn"].unique())
print(f"Fechas de llegada: {len(fechas)}")

for fecha in fechas:
    reservas_fecha = df_modelo[df_modelo["CheckIn"] == fecha]
    ocupacion_real = len(reservas_fecha)
    adr_final = reservas_fecha["ADR"].mean()
    for dias_vista in HORIZONTES:
        fecha_consulta = fecha - pd.Timedelta(days=dias_vista)
        reservas_vigentes = reservas_fecha[reservas_fecha["FechaReserva"] <= fecha_consulta]
        habitaciones_vendidas = len(reservas_vigentes)
        ocupacion_actual = habitaciones_vendidas / CAPACIDAD_HOTEL * 100
        adr_actual = reservas_vigentes["ADR"].mean() if len(reservas_vigentes) > 0 else 0
        lead_time = reservas_vigentes["LeadTime"].mean() if len(reservas_vigentes) > 0 else dias_vista

        canal_moda = reservas_vigentes["Canal"].mode()[0] if not reservas_vigentes.empty else 'Otros'
        tipo_moda = reservas_vigentes["Tipo"].mode()[0] if not reservas_vigentes.empty else 'Otros'
        pais_moda = reservas_vigentes["Pais"].mode()[0] if not reservas_vigentes.empty else 'Otros'

        dataset_modelo.append({
            "CheckIn": fecha, "dias_vista": dias_vista,
            "habitaciones_vendidas": habitaciones_vendidas,
            "ocupacion_actual": ocupacion_actual, "adr_actual": adr_actual,
            "lead_time": lead_time,
            "ocupacion_final": ocupacion_real / CAPACIDAD_HOTEL * 100,
            "adr_final": adr_final,
            "Canal": canal_moda, "Tipo": tipo_moda, "Pais": pais_moda,
        })

dataset_modelo = pd.DataFrame(dataset_modelo)
dataset_modelo = pd.get_dummies(dataset_modelo, columns=["Canal", "Tipo", "Pais"], drop_first=True, dtype=int)
print(dataset_modelo.shape)
display(dataset_modelo.head())

pickup_7 = []
for _, fila in dataset_modelo.iterrows():
    fecha = fila["CheckIn"]
    fecha_consulta = fecha - pd.Timedelta(days=fila["dias_vista"])
    inicio = fecha_consulta - pd.Timedelta(days=7)
    reservas = df_modelo[(df_modelo["CheckIn"] == fecha) & (df_modelo["FechaReserva"] > inicio) & (df_modelo["FechaReserva"] <= fecha_consulta)]
    pickup_7.append(len(reservas))
dataset_modelo["pickup_7"] = pickup_7

pickup_14 = []
for _, fila in dataset_modelo.iterrows():
    fecha = fila["CheckIn"]
    fecha_consulta = fecha - pd.Timedelta(days=fila["dias_vista"])
    inicio = fecha_consulta - pd.Timedelta(days=14)
    reservas = df_modelo[(df_modelo["CheckIn"] == fecha) & (df_modelo["FechaReserva"] > inicio) & (df_modelo["FechaReserva"] <= fecha_consulta)]
    pickup_14.append(len(reservas))
dataset_modelo["pickup_14"] = pickup_14

print("pickup_7 / pickup_14 calculados")

historico = ocupacion_final.copy()
historico["CheckIn"] = historico["CheckIn"] + pd.DateOffset(years=1)
historico = historico.rename(columns={"ocupacion_final": "ocupacion_anio_anterior"})
dataset_modelo = dataset_modelo.merge(historico[["CheckIn", "ocupacion_anio_anterior"]], on="CheckIn", how="left")
dataset_modelo["ocupacion_anio_anterior"] = dataset_modelo["ocupacion_anio_anterior"].fillna(dataset_modelo["ocupacion_actual"])

historico_adr = ocupacion_final.copy()
historico_adr["CheckIn"] = historico_adr["CheckIn"] + pd.DateOffset(years=1)
historico_adr = historico_adr.rename(columns={"adr_final": "adr_anio_anterior"})
dataset_modelo = dataset_modelo.merge(historico_adr[["CheckIn", "adr_anio_anterior"]], on="CheckIn", how="left")
dataset_modelo["adr_anio_anterior"] = dataset_modelo["adr_anio_anterior"].fillna(dataset_modelo["adr_actual"])

dataset_modelo["Mes"] = dataset_modelo["CheckIn"].dt.month
dataset_modelo["Semana"] = dataset_modelo["CheckIn"].dt.isocalendar().week.astype(int)
dataset_modelo["DiaMes"] = dataset_modelo["CheckIn"].dt.day
dataset_modelo["DiaSemana"] = dataset_modelo["CheckIn"].dt.dayofweek
dataset_modelo["FinSemana"] = (dataset_modelo["DiaSemana"] >= 5).astype(int)
dataset_modelo["Año"] = dataset_modelo["CheckIn"].dt.year

es = holidays.ES()
dataset_modelo["Festivo"] = dataset_modelo["CheckIn"].dt.date.isin(es).astype(int)

dataset_modelo["Es_Verano"] = dataset_modelo["Mes"].isin([6, 7, 8]).astype(int)
dataset_modelo["Es_Invierno"] = dataset_modelo["Mes"].isin([12, 1, 2]).astype(int)
dataset_modelo["Es_Puente"] = 0

dataset_modelo["FinMes"] = (dataset_modelo["DiaMes"] >= 25).astype(int)
dataset_modelo["InicioMes"] = (dataset_modelo["DiaMes"] <= 5).astype(int)

dataset_modelo["Mes_sin"] = np.sin(2*np.pi*dataset_modelo["Mes"]/12)
dataset_modelo["Mes_cos"] = np.cos(2*np.pi*dataset_modelo["Mes"]/12)
dataset_modelo["DiaSemana_sin"] = np.sin(2*np.pi*dataset_modelo["DiaSemana"]/7)
dataset_modelo["DiaSemana_cos"] = np.cos(2*np.pi*dataset_modelo["DiaSemana"]/7)

display(dataset_modelo.head())

In [ ]:
df["Tipo"].value_counts()

In [ ]:
df["Canal"].value_counts()

In [ ]:
dataset_modelo.columns.tolist()


### ----------------------------- BLOQUE 4 — Limpieza y validacion -----------------------------

In [ ]:
dataset_modelo = dataset_modelo.fillna(0)
dataset_modelo = dataset_modelo.sort_values(["CheckIn", "dias_vista"]).reset_index(drop=True)
print(dataset_modelo.info())

dataset_modelo["nivel_ocupacion"] = pd.cut(
    dataset_modelo["ocupacion_final"], bins=[0, 40, 70, 100],
    labels=["Baja", "Media", "Alta"], include_lowest=True
)
print("Distribución por nivel de ocupación")
display(dataset_modelo["nivel_ocupacion"].value_counts().sort_index())
print(f"\\nDuplicados: {dataset_modelo.duplicated().sum()}")

constantes = [c for c in dataset_modelo.columns if dataset_modelo[c].nunique() == 1]
print("Variables constantes:", "ninguna" if not constantes else constantes)


md("### Guardar `dataset_modelo.csv` en Drive")

os.makedirs(CARPETA_MODELO, exist_ok=True)
file_path_in_drive = os.path.join(CARPETA_MODELO, 'dataset_modelo.csv')
dataset_modelo.to_csv(file_path_in_drive, index=False)
print(f"Guardado en: {file_path_in_drive}")


### ---------------------------- BLOQUE 5 — Entrenamiento ----------------------------
Entrenamiento del modelo XGBoost

In [ ]:
dataset_modelo["pickup"] = dataset_modelo["ocupacion_final"] - dataset_modelo["ocupacion_actual"]
objetivo = "pickup"

excluir = [
    "CheckIn", "ocupacion_final", "adr_final", "nivel_ocupacion",
    "Año", "habitaciones_vendidas", "Mes", "DiaSemana", "Es_Puente", "pickup"
]
variables = [c for c in dataset_modelo.columns if c not in excluir]

X = dataset_modelo[variables]
y = dataset_modelo[objetivo]
print(f"Variables predictoras: {len(variables)}")
display(X.head())

train = dataset_modelo["Año"] < 2025
test = dataset_modelo["Año"] == 2025

X_train, X_test = X.loc[train], X.loc[test]
y_train, y_test = y.loc[train], y.loc[test]

print(f"Train: {len(X_train):,}")
print(f"Test : {len(X_test):,}")

assert X_train.isnull().sum().sum() == 0
assert X_test.isnull().sum().sum() == 0
print("Datos preparados correctamente.")

modelo = XGBRegressor(
    objective="reg:squarederror", n_estimators=300, learning_rate=0.05,
    max_depth=5, subsample=0.8, colsample_bytree=0.8, random_state=RANDOM_STATE
)
modelo.fit(X_train, y_train)
print("Modelo entrenado correctamente.")

os.makedirs(CARPETA_MODELO, exist_ok=True)
modelo_path = os.path.join(CARPETA_MODELO, "modelo_ocupacion.pkl")
variables_path = os.path.join(CARPETA_MODELO, "variables_modelo.pkl")

joblib.dump(modelo, modelo_path)
joblib.dump(variables, variables_path)

print("Archivos guardados correctamente:")
print(f"- Modelo   : {modelo_path}")
print(f"- Variables: {variables_path}")

pred_pickup_train = np.maximum(0, modelo.predict(X_train))
pred_pickup_test = np.maximum(0, modelo.predict(X_test))

pred_train_final = X_train["ocupacion_actual"] + pred_pickup_train
pred_test_final = X_test["ocupacion_actual"] + pred_pickup_test
y_test_final = X_test["ocupacion_actual"] + y_test

print(f"Predicciones Train: {len(pred_train_final):,}")
print(f"Predicciones Test : {len(pred_test_final):,}")

mae = mean_absolute_error(y_test_final, pred_test_final)
rmse = np.sqrt(mean_squared_error(y_test_final, pred_test_final))
mape = np.mean(np.abs((y_test_final - pred_test_final) / y_test_final.replace(0, np.nan))) * 100
r2 = r2_score(y_test_final, pred_test_final)

metricas = pd.DataFrame({"Métrica": ["MAE", "RMSE", "MAPE", "R²"], "Valor": [mae, rmse, mape, r2]})
display(metricas)

plt.figure(figsize=(7, 7))
plt.scatter(y_test_final, pred_test_final, alpha=0.6)
plt.plot([0, 100], [0, 100], "r--")
plt.xlabel("Ocupación real (%)")
plt.ylabel("Ocupación predicha (%)")
plt.title("Predicción vs Real")
plt.grid(True)
plt.show()

errores = pred_test_final - y_test_final
plt.figure(figsize=(8, 5))
plt.scatter(pred_test_final, errores, alpha=0.6)
plt.axhline(0, color="red", linestyle="--")
plt.xlabel("Predicción")
plt.ylabel("Residual")
plt.title("Análisis de residuales")
plt.grid(True)
plt.show()

importancias = pd.DataFrame({
    "Variable": variables, "Importancia": modelo.feature_importances_
}).sort_values("Importancia", ascending=False)
display(importancias)

plt.figure(figsize=(8, 6))
plt.barh(importancias["Variable"], importancias["Importancia"])
plt.gca().invert_yaxis()
plt.title("Importancia de variables")
plt.tight_layout()
plt.show()

explainer = shap.TreeExplainer(modelo)
shap_values = explainer.shap_values(X_test)
shap.summary_plot(shap_values, X_test)

print("="*60)
print("RESUMEN DEL MODELO")
print("="*60)
print(f"MAE  : {mae:.2f}")
print(f"RMSE : {rmse:.2f}")
print(f"MAPE : {mape:.2f}%")
print(f"R²   : {r2:.4f}")
print("\\nTop 10 variables:")
display(importancias.head(10))


### ---------------------------- BLOQUE 6 — Despliegue automatico ----------------------------
Despliegue automático a Cloud Functions

In [ ]:
RUTA_MAIN_PY = '/content/drive/MyDrive/Master/TFM/Modelos TFM/main.py'
RUTA_REQUIREMENTS = '/content/drive/MyDrive/Master/TFM/Modelos TFM/requirements.txt'

for ruta in [RUTA_MAIN_PY, RUTA_REQUIREMENTS]:
    if not os.path.exists(ruta):
        print(f"⚠ No encuentro {ruta} — súbelo con el icono de archivos de la izquierda, o ajusta la ruta.")

from google.colab import auth
auth.authenticate_user()
print("Autenticado ✅")

!gcloud config set project {GCP_PROJECT_ID}
!gcloud config get-value project

import shutil

if os.path.exists(CARPETA_DEPLOY):
    shutil.rmtree(CARPETA_DEPLOY)
os.makedirs(CARPETA_DEPLOY)

shutil.copy(modelo_path, os.path.join(CARPETA_DEPLOY, 'modelo_ocupacion.pkl'))
shutil.copy(variables_path, os.path.join(CARPETA_DEPLOY, 'variables_modelo.pkl'))
shutil.copy(RUTA_MAIN_PY, os.path.join(CARPETA_DEPLOY, 'main.py'))
shutil.copy(RUTA_REQUIREMENTS, os.path.join(CARPETA_DEPLOY, 'requirements.txt'))

print("Paquete de despliegue listo en:", CARPETA_DEPLOY)
print(os.listdir(CARPETA_DEPLOY))

!gcloud functions deploy {FUNCTION_NAME} --gen2 --runtime={FUNCTION_RUNTIME} --region={GCP_REGION} --source={CARPETA_DEPLOY} --entry-point={FUNCTION_ENTRY} --trigger-http --memory={FUNCTION_MEMORY} --timeout={FUNCTION_TIMEOUT} --allow-unauthenticated

In [ ]:
import subprocess

resultado = subprocess.run(
    ['gcloud', 'functions', 'describe', FUNCTION_NAME,
     '--region', GCP_REGION, '--gen2',
     '--format=value(serviceConfig.uri)'],
    capture_output=True, text=True)

url = resultado.stdout.strip()
if url:
    print("✅ Despliegue completado.")
    print(f"URL de la Cloud Function:\n{url}")
else:
    print("No se pudo obtener la URL automáticamente. Salida completa:")
    print(resultado.stdout)
    print(resultado.stderr)

In [ ]:
import requests

payload = {
    "data": [{"Fecha": (pd.Timestamp.today() + pd.Timedelta(days=5)).strftime('%Y-%m-%d'),
               "ocupacion_actual": 40}],
    "horizonte": 14}

if url:
    resp = requests.post(url, json=payload, timeout=30)
    print(resp.status_code)
    print(resp.json())
else:
    print("Sin URL disponible todavía — ejecuta la celda anterior primero.")

nb['cells'] = cells
nb['metadata'] = {
    "kernelspec": {"display_name": "Python 3", "language": "python", "name": "python3"},
    "language_info": {"name": "python", "version": "3.11"},
    "colab": {"provenance": []},
}

with open('/home/claude/gen/Pipeline_Modelo_Hotel_Completo.ipynb', 'w') as f:
    nbf.write(nb, f)

print(f"Notebook creado con {len(cells)} celdas")

In [ ]:
import subprocess

resultado = subprocess.run(
    ['gcloud', 'functions', 'describe', FUNCTION_NAME,
     '--region', GCP_REGION, '--gen2',
     '--format=value(serviceConfig.uri)'],
    capture_output=True, text=True)

url = resultado.stdout.strip()
if url:
    print("✅ Despliegue completado.")
    print(f"URL de la Cloud Function:\\n{url}")
else:
    print("No se pudo obtener la URL automáticamente. Salida completa:")
    print(resultado.stdout)
    print(resultado.stderr)


In [ ]:
import requests

payload = {
    "data": [{"Fecha": (pd.Timestamp.today() + pd.Timedelta(days=5)).strftime('%Y-%m-%d'),
               "ocupacion_actual": 40}],
    "horizonte": 14}

if url:
    resp = requests.post(url, json=payload, timeout=30)
    print(resp.status_code)
    print(resp.json())
else:
    print("Sin URL disponible todavía — ejecuta la celda anterior primero.")

nb['cells'] = cells
nb['metadata'] = {
    "kernelspec": {"display_name": "Python 3", "language": "python", "name": "python3"},
    "language_info": {"name": "python", "version": "3.11"},
    "colab": {"provenance": []},
}

with open('/home/claude/gen/Pipeline_Modelo_Hotel_Completo.ipynb', 'w') as f:
    nbf.write(nb, f)

print(f"Notebook creado con {len(cells)} celdas")